# EDA: DDR Retinal Disease Detection Dataset

**Exploratory Data Analysis** for the DDR (Diabetic Retinopathy Detection) dataset.  
This notebook analyzes:

1. Class distribution across train / valid / test
2. Class imbalance ratio
3. File size distribution
4. Image resolution analysis
5. RGB channel brightness
6. Train / Valid / Test split proportions

---

## Setup & Configuration

In [ ]:
from __future__ import annotations

import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

try:
    from tqdm.notebook import tqdm

    tqdm(range(1), disable=True)
except Exception:
    from tqdm import tqdm

%matplotlib inline

# --- Paths ---
PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = PROJECT_ROOT / "reports" / "eda"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# --- Class names (DR severity grades) ---
CLASS_NAMES = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative",
}

SPLITS = ["train", "valid", "test"]

# --- Color palettes ---
COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]
SPLIT_COLORS = ["#3498db", "#2ecc71", "#e74c3c"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Report dir:   {REPORT_DIR}")

## Data Loading

Parse annotation files (`train.txt`, `valid.txt`, `test.txt`).  
Each line: `<filename> <label>` where label is 0-4 (DR severity grade).

In [ ]:
def parse_annotations(split: str) -> list[tuple[str, int]]:
    """Parse annotation file: <filename> <label>."""
    ann_path = DATA_DIR / f"{split}.txt"
    records = []
    with open(ann_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.rsplit(maxsplit=1)
            if len(parts) == 2:
                filename, label = parts[0], int(parts[1])
                records.append((filename, label))
    return records


all_data: dict[str, list[tuple[str, int]]] = {}
for split in SPLITS:
    all_data[split] = parse_annotations(split)
    print(f"  [{split:>5}] {len(all_data[split]):>5} images")

total = sum(len(v) for v in all_data.values())
print(f"  {'TOTAL':>7}: {total:>5} images")

---

## 1. Class Distribution Across Splits

How are DR severity grades distributed in each split?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle(
    "DR Severity Grade Distribution by Split", fontsize=15, fontweight="bold", y=1.02
)

for ax, split in zip(axes, SPLITS):
    labels = [rec[1] for rec in all_data[split]]
    counter = Counter(labels)
    x_labels = [CLASS_NAMES[i] for i in range(5)]
    counts = [counter.get(i, 0) for i in range(5)]

    bars = ax.bar(
        x_labels, counts, color=COLORS, edgecolor="black", linewidth=0.5, alpha=0.9
    )

    for bar, count in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts) * 0.02,
            f"{count}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )

    ax.set_title(f"{split.upper()} ({len(all_data[split])} img)", fontsize=12)
    ax.set_xlabel("DR Grade")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].set_ylabel("Number of Images")
plt.tight_layout()
plt.savefig(REPORT_DIR / "01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Insight #1: Class Distribution

| Class | Train | Valid | Test | Total | % |
|-------|------:|------:|-----:|------:|----:|
| **No DR** | 3133 | 1253 | 1880 | 6266 | 45.8% |
| **Mild** | 315 | 126 | 189 | 630 | 4.6% |
| **Moderate** | 2238 | 895 | 1344 | 4477 | 32.7% |
| **Severe** | 118 | 47 | 71 | 236 | 1.7% |
| **Proliferative** | 456 | 182 | 275 | 913 | 6.7% |

**The dataset is heavily imbalanced.**  
- Almost half (45.8%) are healthy eyes (No DR)  
- A third (32.7%) are Moderate  
- **Severe is only 1.7%** (118 images in train!) — the model will struggle to learn this class  
- Mild is also underrepresented at 4.6%

---

## 2. Imbalance Ratio

How much more data does the largest class have compared to each class?

In [ ]:
train_labels = [rec[1] for rec in all_data["train"]]
train_counter = Counter(train_labels)
max_class_count = max(train_counter.values())

fig, ax = plt.subplots(figsize=(10, 5))
ratios = []
x_labels = []
for cls_id in range(5):
    count = train_counter.get(cls_id, 1)
    ratio = max_class_count / count
    ratios.append(ratio)
    x_labels.append(CLASS_NAMES[cls_id])

bars = ax.barh(
    x_labels, ratios, color=COLORS, edgecolor="black", linewidth=0.5, alpha=0.9
)

for bar, ratio, cls_id in zip(bars, ratios, range(5)):
    count = train_counter.get(cls_id, 0)
    ax.text(
        bar.get_width() + 0.1,
        bar.get_y() + bar.get_height() / 2,
        f"x{ratio:.1f}  ({count} img)",
        va="center",
        fontsize=10,
    )

ax.set_xlabel("Imbalance Ratio (max_class / current_class)")
ax.set_title("Imbalance Ratio by Class (train)", fontsize=13, fontweight="bold")
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(REPORT_DIR / "02_imbalance_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

### Insight #2: Extreme Class Imbalance

- **Severe is 26.6x underrepresented** compared to No DR — this is the most clinically critical class (patients need urgent treatment), yet the model has the least data for it
- **Mild is 9.9x underrepresented** — early detection is crucial, but data is scarce
- **Proliferative is 6.9x underrepresented**
- Only Moderate (x1.4) is somewhat balanced with No DR

**Recommendations:**
- Use **Class-Balanced Loss** or **Focal Loss**
- Apply **WeightedRandomSampler** for oversampling Severe/Mild  
- Monitor **per-class recall** rather than just accuracy  
- Apply extra **augmentations** for rare classes

---

## 3. File Size Distribution

File size proxies for image quality and resolution.

In [ ]:
file_sizes_kb = []
for filename, label in tqdm(all_data["train"], desc="Scanning file sizes"):
    fpath = DATA_DIR / "train" / filename
    if fpath.exists():
        file_sizes_kb.append(fpath.stat().st_size / 1024)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(file_sizes_kb, bins=80, color="#3498db", edgecolor="white", alpha=0.85)
ax.axvline(
    np.median(file_sizes_kb),
    color="#e74c3c",
    linestyle="--",
    linewidth=2,
    label=f"Median: {np.median(file_sizes_kb):.0f} KB",
)
ax.axvline(
    np.mean(file_sizes_kb),
    color="#2ecc71",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {np.mean(file_sizes_kb):.0f} KB",
)
ax.set_xlabel("File Size (KB)")
ax.set_ylabel("Number of Images")
ax.set_title("File Size Distribution (train)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(REPORT_DIR / "03_file_size_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Median: {np.median(file_sizes_kb):.0f} KB")
print(f"Mean:   {np.mean(file_sizes_kb):.0f} KB")
print(f"Min:    {np.min(file_sizes_kb):.0f} KB")
print(f"Max:    {np.max(file_sizes_kb):.0f} KB")

### Insight #3: Bimodal File Size Distribution

- File sizes range from **65 KB to 5073 KB** (5 MB!)
- The distribution is **bimodal** — two clear peaks:
  - Small files (~100-200 KB) — likely from lower-resolution cameras
  - Larger files (~800-1100 KB) — from higher-quality fundus cameras
- **The data comes from multiple devices/clinics** with varying image quality
- The preprocessing pipeline resizes to 512x512, which normalizes this variance

---

## 4. Image Resolution Analysis

Sampling 500 images to check actual pixel dimensions.

In [ ]:
random.seed(42)
sample_size = min(500, len(all_data["train"]))
sample_indices = random.sample(range(len(all_data["train"])), sample_size)

widths = []
heights = []
broken_count = 0

for idx in tqdm(sample_indices, desc="Reading resolutions"):
    filename, label = all_data["train"][idx]
    fpath = DATA_DIR / "train" / filename
    try:
        with Image.open(fpath) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except Exception:
        broken_count += 1

if broken_count > 0:
    print(f"WARNING: {broken_count} broken files in sample")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Image Resolutions (sample of 500 from train)",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)

# Scatter: width vs height
axes[0].scatter(widths, heights, alpha=0.4, s=15, color="#8e44ad", edgecolors="none")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")
axes[0].set_title("Width x Height")
axes[0].grid(linestyle="--", alpha=0.4)

# Top-10 resolutions bar chart
unique_resolutions = Counter(zip(widths, heights))
res_labels = [f"{w}x{h}" for (w, h), _ in unique_resolutions.most_common(10)]
res_counts = [c for _, c in unique_resolutions.most_common(10)]
axes[1].barh(
    res_labels,
    res_counts,
    color="#e67e22",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.85,
)
axes[1].set_xlabel("Count (out of 500)")
axes[1].set_title("Top-10 Resolutions")
axes[1].invert_yaxis()
axes[1].grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(REPORT_DIR / "04_image_resolutions.png", dpi=150, bbox_inches="tight")
plt.show()

uw = sorted(set(widths))
uh = sorted(set(heights))
print(f"Unique widths:  {len(uw)}, range: {min(uw)}-{max(uw)} px")
print(f"Unique heights: {len(uh)}, range: {min(uh)}-{max(uh)} px")

### Insight #4: Highly Heterogeneous Resolutions

- **38+ unique resolutions** ranging from 1380x1088 to 4496x3456 pixels
- Top-3 most common: 2048x1536, 2592x1728, 2736x1824
- The scatter plot shows a **cloud** of points, not a single cluster
- **Images come from many different camera models**
- Resize to 512x512 is mandatory for batching, but compressing 4496px -> 512px loses fine-grained details (microaneurysms, small hemorrhages)
- Consider increasing input resolution to 1024x1024 for better detail preservation

---

## 5. RGB Channel Brightness

Analyzing pixel intensity distributions to understand color characteristics of fundus images.

In [ ]:
random.seed(42)
brightness_sample = min(300, len(all_data["train"]))
brightness_indices = random.sample(range(len(all_data["train"])), brightness_sample)

r_means = []
g_means = []
b_means = []

for idx in tqdm(brightness_indices, desc="Computing channel brightness"):
    filename, label = all_data["train"][idx]
    fpath = DATA_DIR / "train" / filename
    try:
        with Image.open(fpath) as img:
            img_rgb = img.convert("RGB")
            arr = np.array(img_rgb, dtype=np.float32)
            r_means.append(arr[:, :, 0].mean())
            g_means.append(arr[:, :, 1].mean())
            b_means.append(arr[:, :, 2].mean())
    except Exception:
        pass

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle(
    "Mean Channel Brightness Distribution (sample of 300)",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)

channel_data = [
    (r_means, "Red", "#e74c3c"),
    (g_means, "Green", "#2ecc71"),
    (b_means, "Blue", "#3498db"),
]

for ax, (data, name, color) in zip(axes, channel_data):
    ax.hist(data, bins=40, color=color, edgecolor="white", alpha=0.8)
    mean_val = np.mean(data)
    ax.axvline(
        mean_val,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"Mean: {mean_val:.1f}",
    )
    ax.set_xlabel(f"{name} channel (0-255)")
    ax.set_title(name)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].set_ylabel("Number of Images")
plt.tight_layout()
plt.savefig(REPORT_DIR / "05_channel_brightness.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean R: {np.mean(r_means):.1f}  (normalized: {np.mean(r_means) / 255:.4f})")
print(f"Mean G: {np.mean(g_means):.1f}  (normalized: {np.mean(g_means) / 255:.4f})")
print(f"Mean B: {np.mean(b_means):.1f}  (normalized: {np.mean(b_means) / 255:.4f})")

### Insight #5: Fundus-Typical Color Profile

- **R=77.7, G=47.7, B=24.1** — Red channel dominates, Blue is very dark
- This is characteristic of **fundus photography**: the retina background is red-orange due to blood vessels
- Dataset-specific normalization values (R~0.30, G~0.19, B~0.09) are **very different from ImageNet** (0.485, 0.456, 0.406)
- Using ImageNet normalization is fine for **transfer learning** with pretrained backbones
- The **Green channel** is the most informative for DR: microaneurysms, exudates, and hemorrhages are best visible there
- Wide spread in Green channel (10-150) indicates good variability for learning

---

## 6. Train / Valid / Test Split Proportions

Is the split stratified? Are class ratios preserved?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Dataset Split: Train / Valid / Test", fontsize=13, fontweight="bold")

# Pie chart
split_sizes = [len(all_data[s]) for s in SPLITS]
split_labels_pie = [
    f"{s.upper()}\n{n} img\n({n / total * 100:.1f}%)"
    for s, n in zip(SPLITS, split_sizes)
]
wedges, texts = axes[0].pie(
    split_sizes,
    labels=split_labels_pie,
    colors=SPLIT_COLORS,
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    textprops=dict(fontsize=10),
)
axes[0].set_title("Overall Split Ratio")

# Stacked bar per class
x = np.arange(5)
width = 0.6
bottoms = np.zeros(5)

for split, color in zip(SPLITS, SPLIT_COLORS):
    labels = [rec[1] for rec in all_data[split]]
    counter = Counter(labels)
    counts = np.array([counter.get(i, 0) for i in range(5)], dtype=float)
    axes[1].bar(
        x,
        counts,
        width,
        bottom=bottoms,
        color=color,
        edgecolor="white",
        linewidth=0.5,
        label=split.upper(),
    )
    bottoms += counts

axes[1].set_xticks(x)
axes[1].set_xticklabels([CLASS_NAMES[i] for i in range(5)], rotation=20)
axes[1].set_ylabel("Number of Images")
axes[1].set_title("Class Distribution per Split")
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(REPORT_DIR / "06_split_proportions.png", dpi=150, bbox_inches="tight")
plt.show()

### Insight #6: Stratified Split

- Split proportions: **50% / 20% / 30%** (train / valid / test)
- Class ratios are **identical** across all three splits (45.8% / 4.6% / 32.7% / 1.7% / 6.7%)
- This confirms **stratified splitting** — validation and test metrics will be representative
- Test set (30%) is larger than valid (20%) — common in medical datasets for reliable final evaluation

---

## Summary

### Data Analysis Key Findings

- **Total dataset**: 13,673 fundus images across 5 DR severity grades (0-4)
- **Critical class imbalance**: Severe (Grade 3) has only 118 train images, **26.6x less** than No DR. Mild is **9.9x underrepresented**
- **Multi-source data**: 38+ unique resolutions (1380px to 4496px), bimodal file size distribution (65 KB to 5 MB) confirm data collected from multiple clinics/cameras
- **Fundus-typical color profile**: R=77.7 >> G=47.7 >> B=24.1; very different from ImageNet statistics, but ImageNet normalization is appropriate for transfer learning
- **Stratified splits**: Class proportions preserved across train/valid/test (50/20/30%)
- **Green channel is most informative** for DR detection due to best contrast for pathological features

### Insights & Next Steps

- **Address class imbalance urgently**: Without intervention (class weights, focal loss, oversampling), the model will underperform on the most clinically important classes (Severe, Mild). Consider WeightedRandomSampler and monitor per-class recall.
- **Consider higher input resolution**: Current config uses 512x512, but many source images are 2048+ px. Testing 1024x1024 could improve detection of small pathological features (microaneurysms, exudates) at the cost of training speed.